# P04A — Explicabilidad tabular en Velora Finance

Velora Finance ya ha evaluado:

- rendimiento global;
- distribución de errores;
- fairness;
- privacidad.

Ahora analizaremos **por qué el modelo produce sus predicciones**.

Trabajaremos con cuatro perspectivas complementarias:

1. **interpretabilidad por diseño** mediante una regresión logística;
2. **SHAP** para explicaciones globales y locales de un modelo XGBoost;
3. **LIME** como aproximación local model-agnostic;
4. **contrafactuales** con DiCE.

La pregunta principal no es “¿qué librería explica mejor?”, sino:

> ¿Qué afirmaciones podemos sostener —y cuáles no— a partir de cada tipo de explicación?


## 1. Instalar las librerías

Usaremos versiones concretas para reproducibilidad:

- SHAP 0.54.1
- LIME 0.2.0.1
- DiCE 0.12
- XGBoost 3.4.1


In [ ]:
!pip -q install shap==0.54.1 lime==0.2.0.1 dice-ml==0.12 xgboost==3.4.1


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    recall_score,
)

from xgboost import XGBClassifier

import shap
from lime.lime_tabular import LimeTabularExplainer
import dice_ml

pd.set_option("display.max_columns", None)

print("SHAP:", shap.__version__)


## 2. Cargar el mismo caso Velora

Mantenemos las mismas variables utilizadas en P01–P03.

`customer_group` continúa fuera del modelo: no es una feature predictora.


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"

namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()

FEATURES = [
    "age",
    "annual_income",
    "debt_ratio",
    "employment_months",
    "late_payments_12m",
    "credit_history_months",
    "existing_debt",
    "requested_amount",
    "savings_balance",
]

X = df[FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Dataset:", df.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("customer_group es predictor:", "customer_group" in FEATURES)


## 3. Dos modelos: interpretable y más opaco

Entrenamos:

- una **regresión logística**, cuya estructura permite inspeccionar directamente coeficientes;
- un **XGBoost**, capaz de representar relaciones no lineales e interacciones, pero cuya decisión no puede resumirse con una sola ecuación sencilla.

No buscamos maximizar el rendimiento. Queremos disponer de dos modelos suficientemente razonables para comparar formas de explicación.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

xgb_model = XGBClassifier(
    n_estimators=450,
    max_depth=3,
    learning_rate=0.035,
    subsample=0.90,
    colsample_bytree=0.90,
    min_child_weight=3,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

logistic_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

def model_metrics(name, model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.50).astype(int)

    return {
        "modelo": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "average_precision": average_precision_score(y_test, proba),
        "accuracy": accuracy_score(y_test, pred),
        "recall_impago": recall_score(y_test, pred, zero_division=0),
    }

comparison = pd.DataFrame([
    model_metrics("Logistic Regression", logistic_model),
    model_metrics("XGBoost", xgb_model),
]).set_index("modelo")

display(comparison.style.format("{:.3f}"))


### Interpretación

Un modelo más complejo solo está justificado si el beneficio compensa sus costes:

- explicación;
- validación;
- robustez;
- mantenimiento;
- gobierno.

Una pequeña mejora de ROC-AUC no convierte automáticamente al modelo más complejo en la mejor elección.


## 4. Interpretabilidad por diseño: coeficientes de la regresión logística

Como las variables se estandarizan antes de la regresión:

- coeficiente positivo → aumenta el log-odds estimado de impago;
- coeficiente negativo → lo reduce;
- la magnitud permite comparar el peso relativo dentro de este modelo.

**No estamos midiendo causalidad.**


In [ ]:
coefficients = logistic_model.named_steps["model"].coef_[0]

coef_table = (
    pd.DataFrame({
        "feature": FEATURES,
        "coeficiente": coefficients,
        "magnitud": np.abs(coefficients),
        "direccion": np.where(coefficients >= 0, "↑ riesgo", "↓ riesgo"),
    })
    .sort_values("magnitud", ascending=False)
)

display(coef_table[["feature", "coeficiente", "direccion"]].style.format({
    "coeficiente": "{:+.3f}"
}))


In [ ]:
plot_coef = coef_table.sort_values("coeficiente")

plt.figure(figsize=(8, 5))
plt.barh(plot_coef["feature"], plot_coef["coeficiente"])
plt.axvline(0, linewidth=1)
plt.xlabel("Coeficiente sobre variables estandarizadas")
plt.title("Regresión logística — interpretación global")
plt.show()


### Qué podemos afirmar

La tabla describe **cómo utiliza las variables este modelo lineal**.

No podemos afirmar:

- que una variable cause el impago;
- que el mismo efecto se mantenga fuera de los datos observados;
- que una explicación global describa perfectamente cada caso individual.


# 5. Modelo XGBoost: explicación global con SHAP

Para el modelo de árboles utilizaremos TreeSHAP.

Configuramos:

- un **background** de entrenamiento;
- `feature_perturbation="interventional"`;
- `model_output="probability"`.

Así las contribuciones SHAP se expresan en el mismo espacio de probabilidad que `predict_proba()`.


In [ ]:
background = X_train.sample(n=min(200, len(X_train)), random_state=42)

shap_explainer = shap.TreeExplainer(
    xgb_model,
    data=background,
    feature_perturbation="interventional",
    model_output="probability",
)

X_explain = X_test.sample(n=min(300, len(X_test)), random_state=42)
shap_values = shap_explainer(X_explain)

print("Observaciones explicadas:", len(X_explain))
print("Features:", len(FEATURES))


## 6. Importancia global SHAP

El valor absoluto medio de SHAP responde aproximadamente a:

> ¿Cuánto contribuye una feature, en promedio, a mover la salida del modelo respecto al valor base?

No indica causalidad y no conserva el signo del efecto.


In [ ]:
shap.plots.bar(shap_values, max_display=9, show=False)
plt.title("XGBoost — importancia global SHAP")
plt.show()


## 7. Beeswarm: importancia y dirección

El beeswarm añade dos piezas de información:

- magnitud de la contribución;
- dirección de la contribución para valores altos o bajos de la feature.

Una misma variable puede aumentar el riesgo en unos casos y reducirlo en otros.


In [ ]:
shap.plots.beeswarm(shap_values, max_display=9, show=False)
plt.title("XGBoost — distribución de contribuciones SHAP")
plt.show()


### Precaución con features correlacionadas

SHAP distribuye contribuciones según un esquema de expectativas condicionadas/intervencionales.

Cuando varias variables contienen información parecida, la atribución puede repartirse entre ellas.

> “Importante para el modelo” no equivale a “causa del resultado”.


# 8. Elegir un caso concreto

Selecciona el tipo de caso que quieres explicar.

- `cerca_threshold`: score próximo a 0,50;
- `alto_riesgo`: score elevado;
- `falso_negativo`: impago real que el modelo no detectó;
- `falso_positivo`: marcado como riesgo sin impago real.


In [ ]:
CASE_TYPE = "cerca_threshold" #@param ["cerca_threshold", "alto_riesgo", "falso_negativo", "falso_positivo"]

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_proba >= 0.50).astype(int)
y_test_np = y_test.to_numpy()

def choose_position(case_type):
    if case_type == "alto_riesgo":
        return int(np.argmax(xgb_proba))

    if case_type == "cerca_threshold":
        return int(np.argmin(np.abs(xgb_proba - 0.50)))

    if case_type == "falso_negativo":
        candidates = np.where((y_test_np == 1) & (xgb_pred == 0))[0]
        if len(candidates):
            return int(candidates[np.argmax(xgb_proba[candidates])])

    if case_type == "falso_positivo":
        candidates = np.where((y_test_np == 0) & (xgb_pred == 1))[0]
        if len(candidates):
            return int(candidates[np.argmin(xgb_proba[candidates])])

    return int(np.argmin(np.abs(xgb_proba - 0.50)))

case_pos = choose_position(CASE_TYPE)
case_index = X_test.index[case_pos]
case = X_test.iloc[[case_pos]]

case_info = df.loc[[case_index], ["application_id", "default"]].copy()
case_info["prob_default_xgb"] = xgb_proba[case_pos]
case_info["pred_default_xgb"] = xgb_pred[case_pos]

display(case_info)
display(case)


# 9. Explicación local con SHAP

El waterfall parte del **valor base** del modelo y muestra qué features empujan la probabilidad hacia arriba o hacia abajo para esta solicitud.

Esta es una explicación local: describe un caso, no todo el sistema.


In [ ]:
local_shap = shap_explainer(case)

shap.plots.waterfall(local_shap[0], max_display=9, show=False)
plt.title(f"SHAP local — {df.loc[case_index, 'application_id']}")
plt.show()


## 10. Comprobar la propiedad aditiva

En espacio de probabilidad:

`valor base + suma de contribuciones SHAP ≈ probabilidad del modelo`

Esta comprobación ayuda a entender qué está representando el waterfall.


In [ ]:
base_value = float(np.asarray(local_shap.base_values).reshape(-1)[0])
shap_sum = float(np.asarray(local_shap.values).reshape(1, -1)[0].sum())
reconstructed = base_value + shap_sum
model_probability = float(xgb_model.predict_proba(case)[:, 1][0])

check = pd.Series({
    "valor_base": base_value,
    "suma_SHAP": shap_sum,
    "base + SHAP": reconstructed,
    "predict_proba": model_probability,
    "diferencia": abs(reconstructed - model_probability),
})

display(check.to_frame("valor").style.format("{:.6f}"))


# 11. Explicación local con LIME

LIME no utiliza la estructura interna de XGBoost.

Genera perturbaciones alrededor del caso, consulta al modelo y ajusta un **modelo local sencillo** que intenta aproximar su comportamiento.

La explicación es, por tanto, una **aproximación local**.


In [ ]:
LIME_SAMPLES = 5000 #@param {type:"slider", min:1000, max:10000, step:1000}

lime_explainer = LimeTabularExplainer(
    training_data=X_train.to_numpy(),
    feature_names=FEATURES,
    class_names=["no_impago", "impago"],
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)

lime_exp = lime_explainer.explain_instance(
    data_row=case.iloc[0].to_numpy(),
    predict_fn=xgb_model.predict_proba,
    labels=(1,),
    num_features=len(FEATURES),
    num_samples=LIME_SAMPLES,
)

lime_table = pd.DataFrame(
    lime_exp.as_list(label=1),
    columns=["regla_local", "peso"]
)

print(f"Fidelidad local aproximada (R²): {lime_exp.score:.3f}")
display(lime_table.style.format({"peso": "{:+.4f}"}))


In [ ]:
lime_plot = lime_table.sort_values("peso")

plt.figure(figsize=(8, 5))
plt.barh(lime_plot["regla_local"], lime_plot["peso"])
plt.axvline(0, linewidth=1)
plt.xlabel("Peso del surrogate local")
plt.title("LIME — explicación local")
plt.show()


### Cómo leer LIME

- peso positivo → el surrogate local empuja hacia `impago`;
- peso negativo → empuja hacia `no impago`;
- `score` indica hasta qué punto el modelo lineal local reproduce las predicciones perturbadas.

Una explicación visualmente convincente puede tener una fidelidad local mediocre.


# 12. Estabilidad de LIME

Una explicación local aproximada puede cambiar cuando cambia el muestreo de perturbaciones.

Repetimos la explicación con varias semillas y comparamos las reglas de mayor peso.


In [ ]:
seeds = [7, 21, 42, 99]
rows = []

for seed in seeds:
    explainer_seed = LimeTabularExplainer(
        training_data=X_train.to_numpy(),
        feature_names=FEATURES,
        class_names=["no_impago", "impago"],
        mode="classification",
        discretize_continuous=True,
        random_state=seed,
    )

    exp_seed = explainer_seed.explain_instance(
        data_row=case.iloc[0].to_numpy(),
        predict_fn=xgb_model.predict_proba,
        labels=(1,),
        num_features=5,
        num_samples=LIME_SAMPLES,
    )

    for rank, (rule, weight) in enumerate(exp_seed.as_list(label=1), start=1):
        rows.append({
            "seed": seed,
            "fidelidad_R2": exp_seed.score,
            "rank": rank,
            "regla": rule,
            "peso": weight,
        })

lime_stability = pd.DataFrame(rows)
display(lime_stability.style.format({
    "fidelidad_R2": "{:.3f}",
    "peso": "{:+.4f}",
}))


### Preguntas

- ¿Se mantienen las mismas variables entre semillas?
- ¿Cambia el orden?
- ¿Cambia el signo de alguna contribución?
- ¿La fidelidad local es estable?

La estabilidad forma parte de la calidad de una explicación.


# 13. Contrafactuales: qué tendría que cambiar para modificar la decisión

Una explicación contrafactual responde:

> ¿Qué cambios en la entrada harían que el modelo produjera otra clase?

No es una atribución como SHAP o LIME.

Y, especialmente:

> **un contrafactual del modelo no es automáticamente una recomendación causal para la persona.**


In [ ]:
# Elegimos una solicitud que el XGBoost clasifique como riesgo,
# preferentemente próxima a 0.60 para que el cambio sea alcanzable.
high_risk_positions = np.where(xgb_proba >= 0.50)[0]

if len(high_risk_positions) == 0:
    raise RuntimeError("No hay casos clasificados como impago con threshold 0.50.")

cf_pos = int(
    high_risk_positions[
        np.argmin(np.abs(xgb_proba[high_risk_positions] - 0.60))
    ]
)

cf_index = X_test.index[cf_pos]
cf_case = X_test.iloc[[cf_pos]].copy()

print("Solicitud:", df.loc[cf_index, "application_id"])
print("Probabilidad original:", f"{xgb_proba[cf_pos]:.3f}")
display(cf_case)


## 14. Restricciones de factibilidad

No permitimos que DiCE cambie cualquier variable.

En este escenario consideramos modificables:

- `requested_amount`;
- `existing_debt`;
- `debt_ratio`;
- `savings_balance`.

Mantenemos fijas variables históricas como edad, antigüedad, pagos tardíos e historial de crédito.

Además restringimos la dirección de algunos cambios para evitar escenarios absurdos.


In [ ]:
ACTIONABLE_FEATURES = [
    "requested_amount",
    "existing_debt",
    "debt_ratio",
    "savings_balance",
]

q01 = X_train.quantile(0.01)
q99 = X_train.quantile(0.99)
original = cf_case.iloc[0]

permitted_range = {
    "requested_amount": [
        float(max(q01["requested_amount"], original["requested_amount"] * 0.40)),
        float(original["requested_amount"]),
    ],
    "existing_debt": [
        float(max(0, q01["existing_debt"])),
        float(original["existing_debt"]),
    ],
    "debt_ratio": [
        float(max(0.02, q01["debt_ratio"])),
        float(original["debt_ratio"]),
    ],
    "savings_balance": [
        float(original["savings_balance"]),
        float(max(original["savings_balance"], q99["savings_balance"])),
    ],
}

display(pd.DataFrame(permitted_range, index=["min", "max"]).T)


## 15. Generar contrafactuales con DiCE

DiCE buscará varias alternativas que:

- cambien la predicción hacia `default = 0`;
- modifiquen únicamente las variables permitidas;
- respeten los rangos definidos;
- intenten mantener cambios relativamente pequeños y diversos.


In [ ]:
dice_data = dice_ml.Data(
    dataframe=df[FEATURES + ["default"]],
    continuous_features=FEATURES,
    outcome_name="default",
)

dice_model = dice_ml.Model(
    model=xgb_model,
    backend="sklearn",
    model_type="classifier",
)

dice_explainer = dice_ml.Dice(
    dice_data,
    dice_model,
    method="genetic",
)

TOTAL_CFS = 3 #@param {type:"slider", min:1, max:5, step:1}

cf_result = dice_explainer.generate_counterfactuals(
    cf_case,
    total_CFs=TOTAL_CFS,
    desired_class=0,
    features_to_vary=ACTIONABLE_FEATURES,
    permitted_range=permitted_range,
    proximity_weight=0.5,
    sparsity_weight=0.5,
    diversity_weight=1.0,
    maxiterations=300,
)

cf_result.visualize_as_dataframe(show_only_changes=True)


### Interpretación de los contrafactuales

Un resultado como:

> reducir el importe solicitado + reducir deuda + aumentar ahorro

significa:

> **el modelo cambiaría su clasificación bajo ese escenario.**

No significa:

> “si la persona hace esto, sabemos causalmente que no incumplirá”.

La factibilidad real exige además:

- coherencia entre variables;
- restricciones de negocio;
- tiempo;
- causalidad;
- ausencia de discriminación indirecta.


# 16. Cuatro tipos de explicación, cuatro preguntas

| Enfoque | Pregunta principal | Alcance |
|---|---|---|
| Coeficientes | ¿Cómo está construido el modelo lineal? | Global / intrínseco |
| SHAP | ¿Qué contribuciones asigna el modelo? | Global y local |
| LIME | ¿Qué surrogate aproxima el comportamiento cerca de este caso? | Local / aproximado |
| Contrafactual | ¿Qué cambios harían cambiar la decisión del modelo? | Local / escenarios |

No debemos utilizar todas las técnicas por defecto.

La explicación adecuada depende de:

- audiencia;
- riesgo;
- decisión;
- modelo;
- necesidad de fidelidad;
- necesidad de acción.


# 17. Límites de una explicación

Una explicación puede ser:

- local pero no global;
- fiel al modelo pero no causal;
- estable o inestable;
- sensible al background o al muestreo;
- correcta matemáticamente y difícil de interpretar;
- comprensible pero demasiado simplificada.

Por tanto:

> **explicabilidad no equivale automáticamente a confianza.**

Una explicación también debe evaluarse.


# Conclusiones de P04A

- Un modelo interpretable por diseño y un modelo explicado post-hoc no son lo mismo.
- SHAP puede aportar explicaciones globales y locales consistentes con la salida del modelo.
- LIME aproxima localmente el modelo y su estabilidad/fidelidad deben comprobarse.
- Los contrafactuales responden a una pregunta distinta de la atribución.
- Ninguna de estas técnicas demuestra causalidad.
- Una explicación debe evaluarse en relación con su audiencia, propósito y riesgo.

En P04B veremos qué cambia cuando la entrada deja de ser tabular y pasa a ser una imagen.
